# Social Network: Scientific Analysis on the Epstein Files

#### Objectives of the project:
- To apply Natural Language Processing (NLP) and Network Science to the 3-million-page DOJ document dump to identify key hubs and gatekeepers within the social web.

#### Project Roadmap
- Extraction: Use spaCy to perform Named Entity Recognition (NER) to find names of people and organizations.
- Cleaning & Deduplication: Use rapidfuzz to normalize names (e.g., merging "Bill Clinton" and "William Clinton").
- Graph Construction: Infer connections based on document co-occurrence.
- Analysis: Calculate centrality measures to find the most influential nodes.

#### Tools Used
- NLP: spaCy (en_core_web_md)
- Data Manipulation: Pandas
- Entity Matching: RapidFuzz
- Graphing: NetworkX & Matplotlib

In [ ]:
import spacy
import pandas as pd
from rapidfuzz import fuzz, process 


nlp = spacy.load("en_core_web_md") # en_core_web_md is a medium-sized English model that includes word vectors and is suitable for various NLP tasks.

In [25]:
text = "Donald Trump and Bill Clinton were seen boarding the Lolita Express at Teterboro Airport in New Jersey."

doc = nlp(text) # this is where the text is processed and stored in an OOP manner

for ent in doc.ents: # we loop through the entities found in the doc
    print(ent.text, "--", ent.label_) # this will print the entity text and its label

Donald Trump -- PERSON
Bill Clinton -- PERSON
the Lolita Express -- ORG
Teterboro Airport -- FAC
New Jersey -- GPE


In [ ]:
result =[]

for ent in doc.ents:
    temp = [ent.text, ent.label_]
    result.append(temp)

df = pd.DataFrame(result, columns=['Entity', 'Type'])

df.head()


,Entity,Type
0,Donald Trump,PERSON
1,Bill Clinton,PERSON
2,the Lolita Express,ORG
3,Teterboro Airport,FAC
4,New Jersey,GPE


In [ ]:
df = df.assign(Normalized_Entity=df['Entity'].str.lower()) # we create a new column with the entity text in lowercase for better matching

df.Normalized_Entity.nunique()

df

,Entity,Type,Normalized_Entity
0,Donald Trump,PERSON,donald trump
1,Bill Clinton,PERSON,bill clinton
2,the Lolita Express,ORG,the lolita express
3,Teterboro Airport,FAC,teterboro airport
4,New Jersey,GPE,new jersey


In [32]:
unique_names = df['Normalized_Entity'].unique().tolist() # we get the unique normalized entity names as a list

query = 'trump ' # this is the name we want to match

matches = process.extract(query, unique_names, limit=3, scorer=fuzz.partial_ratio) # we use rapidfuzz's process.extract to find the best matches for the query in the unique names, using partial_ratio as the scoring function

print(f"Matches for '{query}': {matches}") # this will print the matches and their scores

Matches for 'trump ': [('donald trump', 90.9090909090909, 0), ('the lolita express', 40.0, 2), ('teterboro airport', 36.36363636363637, 3)]


## Initialization of the Social Web

In [33]:
# A fake "page" from the files
document_text = """
Ghislaine Maxwell was seen with Jeffrey Epstein at the palm beach residence. 
Also present was Donald Trump, who arrived later that evening.
"""

# 1. Process the text with your 'nlp' object
doc = nlp(document_text)

# 2. Extract only the PERSON names into a list
# Hint: Use a list comprehension to get [ent.text for ent in doc.ents if ent.label_ == 'PERSON']
page_entities = [ent.text.lower().strip() for ent in doc.ents if ent.label_ == 'PERSON']

print(f"Entities on this page: {page_entities}")

Entities on this page: ['ghislaine maxwell', 'jeffrey epstein', 'donald trump']
